In [2]:
import sys; sys.path.append("../src")
import pandas as pd, numpy as np, time
from common import clean_name, core_name, clean_addr, addr_numbers

s1 = pd.read_pickle("../work/tr_s1.pkl")
s2 = pd.read_pickle("../work/tr_s2.pkl")
s3 = pd.read_pickle("../work/tr_s3.pkl")
gt = pd.read_pickle("../work/tr_gt.pkl")

# ---- 1. Sample: 50k S1 + their true matches + some "no owner" records ----
s1s = s1.sample(50000, random_state=42)
g = gt[gt.matched_entity_ids != ""].copy()
g["mid"] = g.matched_entity_ids.str.split(",")
all_pairs = g[["source1_entity_id", "mid"]].explode("mid")
true_pairs = all_pairs[all_pairs.source1_entity_id.isin(s1s.entity_id)]
print("True pairs in sample:", len(true_pairs))

other = pd.concat([s2, s3])
matched_any = set(all_pairs.mid)
q_true = other[other.entity_id.isin(set(true_pairs.mid))]
q_none = other[~other.entity_id.isin(matched_any)].sample(len(q_true) // 3, random_state=42)
q = pd.concat([q_true, q_none])
del other
print("S1 records:", len(s1s), "| S2/S3 records to match:", len(q))

# ---- 2. Clean both sides ----
def prep(df):
    df = df.copy()
    df["name"] = [clean_name(x) for x in df.business_name]
    df["core"] = [core_name(x) for x in df["name"]]
    df["addr"] = [clean_addr(a, c) for a, c in zip(df.business_address, df.country)]
    df["nums"] = [addr_numbers(x) for x in df["addr"]]
    return df

t = time.time()
s1c, qc = prep(s1s), prep(q)
print("Cleaning time (s):", round(time.time() - t, 1))

# ---- 3. Build keys ----
def add_keys(df):
    w = df.core.str.split()
    df["k_first"]  = df.country + "|" + w.str[0].fillna("")
    df["k_long"]   = df.country + "|" + w.apply(lambda x: max(x, key=len) if x else "")
    df["k_num3"]   = df.country + "|" + df.nums.str.split().str[0].fillna("") + "|" + df.core.str[:3]
    df["k_sorted"] = df.country + "|" + w.apply(lambda x: " ".join(sorted(x)[:2]) if x else "")
    return df

s1c, qc = add_keys(s1c), add_keys(qc)
KEYS = ["k_first", "k_long", "k_num3", "k_sorted"]

# ---- 4. Measure each key ----
truth = set(zip(true_pairs.mid, true_pairs.source1_entity_id))
MAX_BLOCK = 200   # skip keys shared by too many S1 records

def block(key):
    a = s1c[["entity_id", key]].rename(columns={"entity_id": "s1_id"})
    a = a[~a[key].str.endswith("|")]
    sizes = a[key].value_counts()
    a = a[a[key].isin(sizes[sizes <= MAX_BLOCK].index)]
    m = qc[["entity_id", key]].merge(a, on=key)
    return set(zip(m.entity_id, m.s1_id))

union = set()
for k in KEYS:
    p = block(k)
    union |= p
    print(f"{k:10s} pairs: {len(p):>9,}   recall: {len(p & truth)/len(truth):.3f}")
print(f"{'UNION':10s} pairs: {len(union):>9,}   recall: {len(union & truth)/len(truth):.3f}")

True pairs in sample: 172636
S1 records: 50000 | S2/S3 records to match: 230181
Cleaning time (s): 9.8
k_first    pairs: 6,217,628   recall: 0.772
k_long     pairs: 6,314,411   recall: 0.618
k_num3     pairs:   305,725   recall: 0.609
k_sorted   pairs:   403,937   recall: 0.588
UNION      pairs: 11,571,062   recall: 0.876


In [3]:
found = pd.DataFrame(list(union), columns=["entity_id", "s1_id"])
tp = true_pairs.rename(columns={"source1_entity_id": "s1_id", "mid": "entity_id"})
miss = tp.merge(found, how="left", indicator=True)
miss = miss[miss["_merge"] == "left_only"].drop(columns="_merge")
miss = miss.merge(qc[["entity_id", "country", "business_name", "core", "addr"]], on="entity_id")
miss = miss.merge(s1c[["entity_id", "business_name", "core", "addr"]]
                  .rename(columns={"entity_id": "s1_id", "business_name": "s1_raw",
                                   "core": "s1_core", "addr": "s1_addr"}), on="s1_id")
print("Missed pairs:", len(miss))
print("By country:", miss.country.value_counts(normalize=True).round(3).to_dict())
for r in miss.sample(30, random_state=1).itertuples():
    print("-" * 80)
    print("S2/S3:", r.business_name, "| core:", r.core, "| addr:", r.addr)
    print("S1   :", r.s1_raw, "| core:", r.s1_core, "| addr:", r.s1_addr)

Missed pairs: 21415
By country: {'India': 0.647, 'US': 0.353}
--------------------------------------------------------------------------------
S2/S3: शिवम बिग प्रोडक्ट्स एलएलपी | core: sivm big prodkts elelpi | addr: no 2nd floor raghav plaza gill colony court road saharanpur uttr prdes
S1   : Shivam Big Products LLP | core: shivam big products | addr: uttar pradesh raghav plaza gill colony court road 2nd floor saharanpur
--------------------------------------------------------------------------------
S2/S3: scholarshipfoundation.com | core: scholarshipfoundation | addr: jarvisburg 912 michael street nc
S1   : Scholarship Foundation Corp | core: scholarship foundation | addr: 112 michael street jarvisburg nc
--------------------------------------------------------------------------------
S2/S3: pediatricspecialists.com | core: pediatricspecialists | addr: 3850 meridian avenue wichita ks
S1   : Pediatric Specialists Inc | core: pediatric specialists | addr: 3850 meridian avenue unit lot

In [1]:
import sys; sys.path.append("../src")
import pandas as pd, time, gc
from common import clean_df

for name in ["tr_s1", "tr_s2", "tr_s3", "te_s1", "te_s2", "te_s3"]:
    t = time.time()
    df = clean_df(pd.read_pickle(f"../work/{name}.pkl"))
    df.to_pickle(f"../work/clean_{name}.pkl")
    print(name, len(df), "rows,", round(time.time() - t), "s")
    del df; gc.collect()

tr_s1 2206821 rows, 40 s
tr_s2 5034616 rows, 134 s
tr_s3 5285603 rows, 177 s
te_s1 1732544 rows, 52 s
te_s2 4887273 rows, 171 s
te_s3 5082316 rows, 167 s


In [ ]:
import sys; sys.path.append("../src")
import pandas as pd, time, os
from blocking import generate_candidates_big

cols = ["entity_id", "country", "core", "addr"]
s1 = pd.read_pickle("../work/clean_te_s1.pkl")[cols]
q = pd.concat([pd.read_pickle("../work/clean_te_s2.pkl")[cols],
               pd.read_pickle("../work/clean_te_s3.pkl")[cols]], ignore_index=True)
print("S1:", len(s1), "| S2+S3:", len(q))

os.makedirs("../work/cand_test", exist_ok=True)
t = time.time()
files = generate_candidates_big(s1, q, "../work/cand_test/part")
print("DONE in", round((time.time() - t) / 60, 1), "minutes |", len(files), "files")

S1: 1732544 | S2+S3: 9969589


In [1]:
import sys; sys.path.append("../src")
import pandas as pd, numpy as np, time
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn

cols = ["entity_id", "country", "core", "addr"]
s1 = pd.read_pickle("../work/clean_tr_s1.pkl")[cols]
s1 = s1[s1.country == "India"].reset_index(drop=True)
other = pd.concat([pd.read_pickle("../work/clean_tr_s2.pkl")[cols],
                   pd.read_pickle("../work/clean_tr_s3.pkl")[cols]])
q = other[other.country == "India"].sample(50000, random_state=5).reset_index(drop=True)
del other

gt = pd.read_pickle("../work/tr_gt.pkl")
g = gt[gt.matched_entity_ids != ""].copy()
g["mid"] = g.matched_entity_ids.str.split(",")
pairs = g[["source1_entity_id", "mid"]].explode("mid")
truth = pairs[pairs.mid.isin(set(q.entity_id))].rename(
    columns={"source1_entity_id": "s1_id", "mid": "entity_id"})
print("India S1:", len(s1), "| records:", len(q), "| true pairs:", len(truth))

def run(col, K, max_df):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 3), min_df=2, max_df=max_df,
                          sublinear_tf=True, dtype=np.float32)
    A = vec.fit_transform(s1[col])
    t = time.time()
    C = sp_matmul_topn(vec.transform(q[col]), A.T.tocsr(), top_n=K, threshold=0.05,
                       sort=True, n_threads=18).tocsr()
    secs = time.time() - t
    rows = np.repeat(np.arange(C.shape[0]), np.diff(C.indptr))
    return pd.DataFrame({"entity_id": q.entity_id.values[rows],
                         "s1_id": s1.entity_id.values[C.indices]}), secs

for mdf in [0.01, 0.003, 0.001]:
    n, t1 = run("core", 10, mdf)
    a, t2 = run("addr", 10, mdf)
    u = pd.concat([n, a]).drop_duplicates()
    rec = len(u.merge(truth, on=["entity_id", "s1_id"])) / len(truth)
    mins_per_million = (t1 + t2) / len(q) * 1e6 / 60
    print(f"max_df {mdf:<6} name {t1:6.0f}s  addr {t2:6.0f}s  -> {mins_per_million:6.1f} min per 1M records"
          f"   recall {rec:.3f}")

India S1: 883188 | records: 50000 | true pairs: 36967
max_df 0.01   name     27s  addr     49s  ->   25.2 min per 1M records   recall 0.868
max_df 0.003  name      3s  addr      7s  ->    3.5 min per 1M records   recall 0.700
max_df 0.001  name      2s  addr      2s  ->    1.4 min per 1M records   recall 0.450


In [2]:
def keep_rarest(B, m):
    """Keep only the m highest-weight (= rarest) pieces of each record."""
    B = B.tocsr().copy()
    for i in range(B.shape[0]):
        s, e = B.indptr[i], B.indptr[i + 1]
        if e - s > m:
            row = B.data[s:e]
            cut = np.partition(row, e - s - m)[e - s - m]
            row[row < cut] = 0
    B.eliminate_zeros()
    return B

def run2(col, K, m):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 3), min_df=2,
                          sublinear_tf=True, dtype=np.float32)
    A = vec.fit_transform(s1[col])
    B = vec.transform(q[col])
    if m is not None:
        B = keep_rarest(B, m)
    t = time.time()
    C = sp_matmul_topn(B, A.T.tocsr(), top_n=K, threshold=0.01,
                       sort=True, n_threads=18).tocsr()
    secs = time.time() - t
    rows = np.repeat(np.arange(C.shape[0]), np.diff(C.indptr))
    return pd.DataFrame({"entity_id": q.entity_id.values[rows],
                         "s1_id": s1.entity_id.values[C.indices]}), secs

for m in [None, 12, 8, 5]:
    n, t1 = run2("core", 10, m)
    a, t2 = run2("addr", 10, m)
    u = pd.concat([n, a]).drop_duplicates()
    rec = len(u.merge(truth, on=["entity_id", "s1_id"])) / len(truth)
    mins_per_million = (t1 + t2) / len(q) * 1e6 / 60
    label = "full" if m is None else f"rarest {m}"
    print(f"{label:<10} name {t1:6.0f}s  addr {t2:6.0f}s  -> {mins_per_million:6.1f} min per 1M   recall {rec:.3f}")

full       name    300s  addr   1283s  ->  527.9 min per 1M   recall 0.938
rarest 12  name    101s  addr    122s  ->   74.4 min per 1M   recall 0.921
rarest 8   name     47s  addr     64s  ->   36.9 min per 1M   recall 0.877
rarest 5   name     17s  addr     25s  ->   14.2 min per 1M   recall 0.763


In [3]:
import importlib, blocking; importlib.reload(blocking)
from blocking import generate_candidates_fast
import os, glob

os.makedirs("../work/tmp_test", exist_ok=True)
t = time.time()
files = generate_candidates_fast(s1, q, "../work/tmp_test/part")
secs = time.time() - t

u = pd.concat([pd.read_pickle(f) for f in files])
rec = len(u.merge(truth, on=["entity_id", "s1_id"])) / len(truth)
print(f"\nFINAL METHOD: {secs:.0f}s -> {secs/len(q)*1e6/60:.1f} min per 1M records | "
      f"pairs per record {len(u)/len(q):.1f} | recall {rec:.3f}")
print("For comparison: full = 0.938 recall at 528 min per 1M")

India rows 0-50,000 of 50,000: 1,232,784 pairs, 78s

FINAL METHOD: 115s -> 38.3 min per 1M records | pairs per record 24.7 | recall 0.929
For comparison: full = 0.938 recall at 528 min per 1M
